In [1]:
import pandas as pd

In [ ]:
actual = pd.read_csv('../data/merged_filter_ingestion.csv')
forecast = pd.read_csv('../data/production_forecast_20251102_032244.csv')

actual['Date'] = pd.to_datetime(actual['Date'])
actual.drop(columns=['Unnamed: 0'], inplace=True)
forecast['Date'] = pd.to_datetime(forecast['Date'])

In [13]:
# Group by Branch, Year, Month and sum Quantity
actual['Year'] = actual['Date'].dt.year
actual['Month'] = actual['Date'].dt.month
branch_monthly_sum_actual = (
    actual.groupby(['Branch', 'Year', 'Month'])['Quantity']
    .sum()
    .reset_index()
    .sort_values(['Branch', 'Year', 'Month'])
)
branch_monthly_sum_actual.head(20)


,Branch,Year,Month,Quantity
0,BLR,2019,4,2667.0
1,BLR,2019,5,2446.5
2,BLR,2019,6,1103.0
3,BLR,2019,7,1222.5
4,BLR,2019,8,1017.0
5,BLR,2019,9,2227.0
6,BLR,2019,10,1413.0
7,BLR,2019,11,1908.5
8,BLR,2019,12,2610.5
9,BLR,2020,1,1919.5


In [17]:
# Group by Branch, Year, Month and sum Quantity
forecast['Year'] = forecast['Date'].dt.year
forecast['Month'] = forecast['Date'].dt.month
branch_monthly_sum_forecast = (
    forecast.groupby(['Branch', 'Year', 'Month'])['forecast']
    .sum()
    .reset_index()
    .sort_values(['Branch', 'Year', 'Month'])
)
branch_monthly_sum_forecast.head(20)


,Branch,Year,Month,forecast
0,BLR,2025,4,2482.204647
1,BLR,2025,5,2240.981798
2,BLR,2025,6,1230.063627
3,BLR,2025,7,609.297859
4,BLR,2025,8,855.049173
5,BLR,2025,9,1477.470265
6,BLR,2025,10,587.058845
7,BLR,2025,11,568.795581
8,BLR,2025,12,1485.854427
9,BLR,2026,1,1340.556253


In [ ]:
branch_monthly_sum_actual[(branch_monthly_sum_actual['Branch'] == 'BLR') & (branch_monthly_sum_actual['Year'] == 2025) & (branch_monthly_sum_actual['Month'] >= 4)]

,Branch,Year,Month,Quantity
71,BLR,2025,4,2829.0
72,BLR,2025,5,2984.0
73,BLR,2025,6,1896.0


In [19]:
# Filter both DataFrames for 2025 months 4, 5, 6 only
actual_2025_q2 = branch_monthly_sum_actual[
    (branch_monthly_sum_actual['Year'] == 2025) &
    (branch_monthly_sum_actual['Month'].isin([4, 5, 6]))
].copy()

forecast_2025_q2 = branch_monthly_sum_forecast[
    (branch_monthly_sum_forecast['Year'] == 2025) &
    (branch_monthly_sum_forecast['Month'].isin([4, 5, 6]))
].copy()

# Merge on Branch, Year, Month
compare_df_q2 = pd.merge(
    actual_2025_q2,
    forecast_2025_q2,
    how='outer',
    on=['Branch', 'Year', 'Month'],
    suffixes=('_actual', '_forecast')
)

# Sort for readability
compare_df_q2 = compare_df_q2.sort_values(['Branch', 'Year', 'Month'])

# Compute error metrics
compare_df_q2['abs_error'] = (compare_df_q2['Quantity'] - compare_df_q2['forecast']).abs()
compare_df_q2['perc_error'] = (
    100 * compare_df_q2['abs_error'] / compare_df_q2['Quantity']
)

# Show comparison
compare_df_q2[['Branch', 'Year', 'Month', 'Quantity', 'forecast', 'abs_error', 'perc_error']]



,Branch,Year,Month,Quantity,forecast,abs_error,perc_error
0,BLR,2025,4,2829.0,2482.204647,346.795353,12.258584
1,BLR,2025,5,2984.0,2240.981798,743.018202,24.900074
2,BLR,2025,6,1896.0,1230.063627,665.936373,35.123226
3,COK,2025,4,6014.0,1816.586336,4197.413664,69.794042
4,COK,2025,5,3941.0,828.247641,3112.752359,78.983820
5,COK,2025,6,2296.0,614.878857,1681.121143,73.219562
6,MAA,2025,4,21001.0,3191.764018,17809.235982,84.801847
7,MAA,2025,5,10334.0,2612.987080,7721.012920,74.714660
8,MAA,2025,6,7101.0,1910.801389,5190.198611,73.091094
9,SBD,2025,4,7847.0,2953.094826,4893.905174,62.366575


In [20]:
# --- Weekly comparison of actual vs forecast ---

# First, ensure we have a proper 'Date' column to make 'Year', 'Week' columns.
# We'll need to aggregate both actual and forecast at weekly level.

# For actuals:
actual = pd.read_csv('../data/merged_filter_ingestion.csv', parse_dates=['Date'])
actual['Year'] = actual['Date'].dt.year
actual['Week'] = actual['Date'].dt.isocalendar().week
# Sum quantity per Branch, Year, Week
weekly_actual = (
    actual.groupby(['Branch', 'Year', 'Week'], as_index=False)['Quantity'].sum()
)

# For forecast:
forecast = pd.read_csv('../data/production_forecast_20251102_032244.csv', parse_dates=['Date'])
forecast['Year'] = forecast['Date'].dt.year
forecast['Week'] = forecast['Date'].dt.isocalendar().week
weekly_forecast = (
    forecast.groupby(['Branch', 'Year', 'Week'], as_index=False)['forecast'].sum()
)

# Align only weeks available in both (optionally focus on a particular year if needed)
merge_weekly = pd.merge(
    weekly_actual, weekly_forecast,
    how='outer',
    on=['Branch', 'Year', 'Week'],
    suffixes=('_actual', '_forecast')
).sort_values(['Branch', 'Year', 'Week'])

# Compute weekly errors
merge_weekly['abs_error'] = (merge_weekly['Quantity'] - merge_weekly['forecast']).abs()
merge_weekly['perc_error'] = 100 * merge_weekly['abs_error'] / merge_weekly['Quantity']

# Show sample of comparison for recent weeks, e.g., 2025 Q2
weekly_2025_q2 = merge_weekly[
    (merge_weekly['Year'] == 2025) & (merge_weekly['Week'].isin([14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]))
]
weekly_2025_q2[['Branch', 'Year', 'Week', 'Quantity', 'forecast', 'abs_error', 'perc_error']].head(20)


,Branch,Year,Week,Quantity,forecast,abs_error,perc_error
307,BLR,2025,14,454.0,NaN,NaN,NaN
308,BLR,2025,15,743.0,1.358478,741.641522,99.817163
309,BLR,2025,16,257.0,1226.980381,969.980381,377.424273
310,BLR,2025,17,499.0,13.904238,485.095762,97.213580
311,BLR,2025,18,1336.0,1239.961550,96.038450,7.188507
312,BLR,2025,19,334.0,13.731308,320.268692,95.888830
313,BLR,2025,20,463.0,1142.658494,679.658494,146.794491
314,BLR,2025,21,427.0,7.424215,419.575785,98.261308
315,BLR,2025,22,1754.0,1077.167781,676.832219,38.587926
316,BLR,2025,23,58.0,3.800027,54.199973,93.448230


In [25]:
weekly_2025_q2[['Branch', 'Year', 'Week', 'Quantity', 'forecast', 'abs_error', 'perc_error']][weekly_2025_q2['Branch'] == 'MAA'].head(20)

,Branch,Year,Week,Quantity,forecast,abs_error,perc_error
1025,MAA,2025,14,4070.0,NaN,NaN,NaN
1026,MAA,2025,15,4630.0,0.000000,4630.000000,100.000000
1027,MAA,2025,16,1837.0,1661.684700,175.315300,9.543566
1028,MAA,2025,17,5639.0,0.000000,5639.000000,100.000000
1029,MAA,2025,18,6565.0,1530.079318,5034.920682,76.693384
1030,MAA,2025,19,1869.0,14.386800,1854.613200,99.230241
1031,MAA,2025,20,1385.0,1213.143789,171.856211,12.408391
1032,MAA,2025,21,2267.0,14.386800,2252.613200,99.365382
1033,MAA,2025,22,4710.0,1371.069690,3338.930310,70.890240
1034,MAA,2025,23,254.0,17.398804,236.601196,93.150077
